# 01 Training of one model (common code)

The six models use exactly this notebook and change only `MODEL` and `MODE`:

| Family | Model | MODEL | MODE | Input |
|---|---|---|---|---|
| CNN | ResNet50 | `resnet50` | `stack` | 2020 + 2024, 16 channels |
| CNN | EfficientNetB0 | `efficientnet_b0` | `stack` | 2020 + 2024, 16 channels |
| Transformer | ViT-B/16 | `vit_b16` | `stack` | 2020 + 2024, 16 channels |
| Transformer | Swin-T | `swin_t` | `stack` | 2020 + 2024, 16 channels |
| Spatiotemporal | CNN+LSTM | `cnn_lstm` | `seq` | 5 dates x 8 channels |
| Spatiotemporal | ConvLSTM | `convlstm` | `seq` | 5 dates x 8 channels |

Settings: GPU T4 and Internet (timm downloads the ImageNet weights). Input: the cache dataset of `00_preprocessing_cache.ipynb`.
First run one epoch (`TEST_RUN = True`), then the full training with Save Version. Output: `/kaggle/working/resultados/<MODEL>/`
with `pred_test_<MODEL>.csv`, `metrics_<MODEL>.json`, `history_<MODEL>.csv`, the two figures and `best_<MODEL>.pt`.

In [ ]:
# Common code (identical for the six models)
import os, glob, json, time, random, numpy as np, pandas as pd
import torch, torch.nn as nn, timm
import matplotlib.pyplot as plt, seaborn as sns
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import accuracy_score, f1_score, precision_recall_fscore_support, confusion_matrix
from statsmodels.stats.proportion import proportion_confint

CFG = dict(epochs=50, batch=16, lr=1e-4, wd=1e-4, patience=8, seed=42)
CLASES = ["No deforestation", "Low", "Moderate", "Severe"]
LOAD_IN_RAM = False      # True reads the 6 GB cache into memory once; use it if Kaggle reads the dataset slowly (same data, same results)
CACHE = glob.glob("/kaggle/input/**/X224.npy", recursive=True)[0]
DIR = os.path.dirname(CACHE); print("cache:", CACHE)
meta = pd.read_csv(f"{DIR}/meta.csv")
stats = json.load(open(f"{DIR}/stats.json"))
X = np.load(CACHE) if LOAD_IN_RAM else np.load(CACHE, mmap_mode="r")
idx = {s: np.where(meta["split"] == s)[0] for s in ["train", "val", "test"]}
print({s: len(v) for s, v in idx.items()})           # must print 1200, 150, 150
DEV = "cuda"

def set_seed(s=CFG["seed"]):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)
    torch.backends.cudnn.deterministic = True; torch.backends.cudnn.benchmark = False

def seed_worker(_):
    s = torch.initial_seed() % 2**32; np.random.seed(s); random.seed(s)

class PatchDS(Dataset):
    def __init__(self, split, mode, train):
        self.ids, self.mode, self.train = idx[split], mode, train
        self.mean = torch.tensor(stats["mean"]).view(1, 8, 1, 1).float()
        self.std = torch.tensor(stats["std"]).view(1, 8, 1, 1).float()
    def __len__(self): return len(self.ids)
    def __getitem__(self, k):
        i = self.ids[k]
        x = (torch.from_numpy(X[i].astype(np.float32)) - self.mean) / self.std
        if self.train:                                   # augmentation only in training, equal for the five dates
            if random.random() < .5: x = x.flip(-1)
            if random.random() < .5: x = x.flip(-2)
            x = torch.rot90(x, random.randint(0, 3), dims=(-2, -1))
        if self.mode == "stack": x = torch.cat([x[0], x[4]], 0)   # 2020 + 2024 = 16 channels
        return x.contiguous(), int(meta["label"].iat[i]), meta["patch_id"].iat[i]

def loaders(mode):
    g = torch.Generator(); g.manual_seed(CFG["seed"])
    def mk(s, tr):
        return DataLoader(PatchDS(s, mode, tr), batch_size=CFG["batch"], shuffle=tr, num_workers=2, pin_memory=True,
                          worker_init_fn=seed_worker, generator=g)
    return mk("train", True), mk("val", False), mk("test", False)

In [ ]:
# Model definitions (identical for the six models)
def build_model(name):
    names = {"resnet50": "resnet50", "efficientnet_b0": "efficientnet_b0",
             "vit_b16": "vit_base_patch16_224", "swin_t": "swin_tiny_patch4_window7_224"}
    if name in names:
        return timm.create_model(names[name], pretrained=True, in_chans=16, num_classes=4)
    if name == "cnn_lstm": return CNNLSTM()
    if name == "convlstm": return ConvLSTMNet()

class CNNLSTM(nn.Module):
    def __init__(self, hidden=256):
        super().__init__()
        self.enc = timm.create_model("efficientnet_b0", pretrained=True, in_chans=8, num_classes=0)
        self.lstm = nn.LSTM(self.enc.num_features, hidden, batch_first=True)
        self.head = nn.Linear(hidden, 4)
    def forward(self, x):                                # x: (B, T, 8, H, W)
        B, T = x.shape[:2]
        f = self.enc(x.flatten(0, 1)).view(B, T, -1)
        _, (h, _) = self.lstm(f)
        return self.head(h[-1])

class ConvLSTMCell(nn.Module):
    def __init__(self, cin, ch, k=3):
        super().__init__(); self.ch = ch
        self.conv = nn.Conv2d(cin + ch, 4 * ch, k, padding=k // 2)
    def forward(self, x, h, c):
        i, f, o, g = torch.chunk(self.conv(torch.cat([x, h], 1)), 4, 1)
        c = torch.sigmoid(f) * c + torch.sigmoid(i) * torch.tanh(g)
        return torch.sigmoid(o) * torch.tanh(c), c

class ConvLSTMNet(nn.Module):
    def __init__(self, ch=64):
        super().__init__()
        self.stem = nn.Sequential(                       # 224 -> 56
            nn.Conv2d(8, 32, 3, 2, 1), nn.BatchNorm2d(32), nn.ReLU(),
            nn.Conv2d(32, 64, 3, 2, 1), nn.BatchNorm2d(64), nn.ReLU())
        self.c1, self.c2 = ConvLSTMCell(64, ch), ConvLSTMCell(ch, ch)
        self.head = nn.Linear(ch, 4)
    def forward(self, x):                                # x: (B, T, 8, H, W)
        B, T = x.shape[:2]
        f = self.stem(x.flatten(0, 1)); f = f.view(B, T, *f.shape[1:])
        h1 = c1 = f.new_zeros(B, self.c1.ch, *f.shape[-2:]); h2 = c2 = h1.clone()
        for t in range(T):
            h1, c1 = self.c1(f[:, t], h1, c1); h2, c2 = self.c2(h1, h2, c2)
        return self.head(h2.mean((-2, -1)))

In [ ]:
# Configuration: change ONLY these two values (see the table above)
MODEL, MODE = "swin_t", "stack"
TEST_RUN = False          # True: one epoch to check that everything runs. False: full training (maximum of 50 epochs)
CFG["epochs"] = 1 if TEST_RUN else 50
print(MODEL, MODE, "| maximum epochs:", CFG["epochs"])

In [ ]:
# Training, evaluation and figures (identical for the six models)
OUT = f"/kaggle/working/resultados/{MODEL}"; os.makedirs(OUT, exist_ok=True)

def run_epoch(model, dl, opt=None, scaler=None):
    train = opt is not None; model.train(train); crit = nn.CrossEntropyLoss()
    tot, ok, n = 0.0, 0, 0
    for x, y, _ in dl:
        x, y = x.to(DEV, non_blocking=True), y.to(DEV)
        with torch.set_grad_enabled(train), torch.amp.autocast("cuda"):
            out = model(x); loss = crit(out, y)
        if train:
            opt.zero_grad(set_to_none=True); scaler.scale(loss).backward()
            scaler.step(opt); scaler.update()
        tot += loss.item() * len(y); ok += (out.argmax(1) == y).sum().item(); n += len(y)
    return tot / n, ok / n

set_seed(); dl_tr, dl_va, dl_te = loaders(MODE)
model = build_model(MODEL).to(DEV)
n_params = sum(p.numel() for p in model.parameters()); print("parameters:", n_params)
opt = torch.optim.AdamW(model.parameters(), lr=CFG["lr"], weight_decay=CFG["wd"])
sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, factor=.5, patience=3)
scaler = torch.amp.GradScaler("cuda")
best, wait, hist, t0 = 1e9, 0, [], time.time()
for ep in range(1, CFG["epochs"] + 1):
    te = time.time()
    tl, ta = run_epoch(model, dl_tr, opt, scaler); vl, va = run_epoch(model, dl_va)
    sched.step(vl)
    hist.append(dict(epoch=ep, train_loss=tl, val_loss=vl, train_acc=ta, val_acc=va, lr=opt.param_groups[0]["lr"]))
    print(f"ep {ep:02d} | train {tl:.4f} {ta:.3f} | val {vl:.4f} {va:.3f} | {time.time()-te:.0f} s", flush=True)
    if vl < best:
        best, wait = vl, 0; torch.save(model.state_dict(), f"{OUT}/best_{MODEL}.pt")
    else: wait += 1
    if wait >= CFG["patience"]: print("early stopping"); break
train_min = (time.time() - t0) / 60
pd.DataFrame(hist).to_csv(f"{OUT}/history_{MODEL}.csv", index=False)

model.load_state_dict(torch.load(f"{OUT}/best_{MODEL}.pt")); model.eval()
ys, ps, probs, pids = [], [], [], []; t1 = time.time()
with torch.no_grad():
    for x, y, pid in dl_te:
        with torch.amp.autocast("cuda"):
            logits = model(x.to(DEV))
        pr = torch.softmax(logits.float(), 1).cpu()
        ys += y.tolist(); ps += pr.argmax(1).tolist(); probs.append(pr); pids += list(pid)
ms_patch = (time.time() - t1) / len(ys) * 1000          # includes reading and transfer of each batch
probs = torch.cat(probs).numpy()
pred = pd.DataFrame({"patch_id": pids, "y_true": ys, "y_pred": ps})
for k in range(4): pred[f"p{k}"] = probs[:, k]
pred.to_csv(f"{OUT}/pred_test_{MODEL}.csv", index=False)

acc = accuracy_score(ys, ps); k_ok = int((np.array(ys) == np.array(ps)).sum())
lo, hi = proportion_confint(k_ok, len(ys), alpha=.05, method="wilson")
P, R, F, _ = precision_recall_fscore_support(ys, ps, labels=[0, 1, 2, 3], zero_division=0)
res = dict(model=MODEL, accuracy=acc, ci95_low=lo, ci95_high=hi, f1_macro=f1_score(ys, ps, average="macro"),
           precision=P.tolist(), recall=R.tolist(), f1=F.tolist(), params=n_params, epochs_run=len(hist),
           train_minutes=train_min, inference_ms_per_patch=ms_patch)
json.dump(res, open(f"{OUT}/metrics_{MODEL}.json", "w"), indent=1); print(res)

h = pd.DataFrame(hist); fig, ax = plt.subplots(1, 2, figsize=(10, 3.6))
ax[0].plot(h.epoch, h.train_loss, label="Training"); ax[0].plot(h.epoch, h.val_loss, label="Validation")
ax[1].plot(h.epoch, h.train_acc, label="Training"); ax[1].plot(h.epoch, h.val_acc, label="Validation")
for a in ax: a.set_xlabel("Epoch"); a.legend()
ax[0].set_ylabel("Loss"); ax[1].set_ylabel("Accuracy")
plt.tight_layout(); plt.savefig(f"{OUT}/fig_curves_{MODEL}.png", dpi=300); plt.show()
cm = confusion_matrix(ys, ps, labels=[0, 1, 2, 3]); plt.figure(figsize=(5.2, 4.4))
sns.heatmap(cm, annot=True, fmt="d", cbar=False, cmap="Blues", xticklabels=CLASES, yticklabels=CLASES)
plt.xlabel("Predicted class"); plt.ylabel("Reference class"); plt.tight_layout()
plt.savefig(f"{OUT}/fig_cm_{MODEL}.png", dpi=300); plt.show()
print("files:", os.listdir(OUT))